# Laboratorio 14 — Transformers y Modelos de Lenguaje
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Noviembre de 2026, por definir · Recuperativa 2 (fuera del programa oficial)
**Bloques:** por definir

En este laboratorio se implementa la autoatención con las propias manos y se mira por dentro un modelo de lenguaje real.

## Objetivos de aprendizaje

1. Calcular a mano la autoatención (Q, K, V) con un ejemplo pequeño.
2. Describir el bloque transformer: atención multicabeza, máscara causal y codificación posicional.
3. Explicar cómo un LLM obtiene la distribución del siguiente token y cómo se muestrea de ella.
4. Explicar por qué un LLM puede equivocarse con total seguridad.

## Plan de la sesión (~110 min)

| Parte | Contenido | Tiempo |
|---|---|---|
| A | Autoatención en NumPy | 30 min |
| B | Multicabeza y codificación posicional | 15 min |
| C | Un modelo real: probabilidades del siguiente token | 35 min |
| D | Mirar la atención real | 20 min |
| E | Clasificación con un modelo preentrenado (opcional) | 10 min |

**Entorno:** Colab (la CPU alcanza). Las Partes A y B funcionan sin conexión; las Partes C, D y E descargan modelos de Hugging Face.

## Configuración

In [ ]:
import sys, random

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q transformers

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Estilo de los gráficos
AZUL, ROSA, GRIS = "#1b2740", "#c8807a", "#8b929c"
plt.rcParams.update({
    "figure.facecolor": "#f2f1ee", "axes.facecolor": "#fffefb",
    "axes.edgecolor": GRIS, "axes.labelcolor": AZUL, "axes.titlecolor": AZUL,
    "text.color": AZUL, "xtick.color": AZUL, "ytick.color": AZUL,
    "axes.titleweight": "bold", "figure.dpi": 110,
    "axes.prop_cycle": plt.cycler(color=[AZUL, ROSA, GRIS]),
})
CMAP = LinearSegmentedColormap.from_list("marino", ["#fffefb", AZUL])


def mapa_calor(matriz, etiquetas_x=None, etiquetas_y=None, titulo="", ax=None, anotar=True, vmax=None):
    """Dibuja una matriz como mapa de calor (filas = consulta, columnas = clave)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(matriz, cmap=CMAP, vmin=0, vmax=vmax if vmax is not None else max(1e-9, float(np.max(matriz))))
    ax.set_title(titulo)
    if etiquetas_x is not None:
        ax.set_xticks(range(len(etiquetas_x)), etiquetas_x, rotation=45, ha="right")
    if etiquetas_y is not None:
        ax.set_yticks(range(len(etiquetas_y)), etiquetas_y)
    if anotar and matriz.shape[0] <= 8:
        for i in range(matriz.shape[0]):
            for j in range(matriz.shape[1]):
                v = matriz[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        color="white" if v > 0.5 * im.norm.vmax else AZUL, fontsize=8)
    return ax

## Parte A — Autoatención en NumPy (30 min)

La fórmula de la atención de producto punto escalado es

$$\text{Atención}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

**Qué hacer.** Implementa en unas 15 líneas:

1. `softmax` por filas (numéricamente estable: restar el máximo de cada fila antes de exponenciar).
2. `atencion(Q, K, V, mascara=None)`, que devuelve la salida y la matriz de pesos.

In [ ]:
def softmax(x, eje=-1):
    """Softmax numéricamente estable a lo largo de `eje`."""
    # TODO: resta el máximo de cada fila (keepdims=True), exponencia y normaliza para que sume 1
    ...


def atencion(Q, K, V, mascara=None):
    """Atención de producto punto escalado.

    Q, K, V: arreglos (n_tokens, d_k).
    mascara: arreglo booleano (n, n); True = se puede mirar esa posición.
    Devuelve (salida, pesos).
    """
    d_k = Q.shape[-1]
    # TODO 1: puntajes = producto punto entre consultas y claves, dividido por la raíz de d_k
    puntajes = ...
    # TODO 2: si hay máscara, deja en -inf (np.inf negativo) las posiciones donde la máscara es False
    ...
    # TODO 3: pesos = softmax por filas de los puntajes
    pesos = ...
    # TODO 4: la salida es el promedio ponderado de los valores V con esos pesos
    salida = ...
    return salida, pesos

### Autocomprobación: el ejemplo de 3 tokens

Tres tokens con embeddings de dimensión 2: `x₁ = (1, 0)`, `x₂ = (0, 1)`, `x₃ = (1, 1)`. Para simplificar, `Q = K = V = X` (en un modelo real son proyecciones aprendidas `X·W_Q`, `X·W_K`, `X·W_V`).

Resultado esperado (el mismo de la clase): pesos `[[0,40 0,20 0,40], [0,20 0,40 0,40], [0,25 0,25 0,50]]` y salida `[[0,80 0,60], [0,60 0,80], [0,75 0,75]]`.

In [ ]:
X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
salida, pesos = atencion(X, X, X)

esperados = np.array([[0.40, 0.20, 0.40], [0.20, 0.40, 0.40], [0.25, 0.25, 0.50]])
assert pesos is not None and np.allclose(pesos, esperados, atol=0.01), "Los pesos no coinciden con los de la clase"
assert np.allclose(pesos.sum(axis=1), 1.0), "Cada fila de pesos debe sumar 1"
assert np.allclose(salida, [[0.80, 0.60], [0.60, 0.80], [0.75, 0.75]], atol=0.01), "La salida no coincide"
print("✓ Ejemplo de 3 tokens reproducido")
print("Pesos:\n", np.round(pesos, 2))
print("Salida:\n", np.round(salida, 2))

etiquetas = ["x₁", "x₂", "x₃"]
mapa_calor(pesos, etiquetas, etiquetas, "Pesos de atención (3 tokens)")
plt.xlabel("token al que se mira (clave)")
plt.ylabel("token que consulta")
plt.tight_layout()
plt.show()

### A.2 — Una frase de 6 tokens con proyecciones aleatorias

Con `d_model = 8` y `d_k = 4`, genera embeddings aleatorios para la frase *«El animal no cruzó la calle»*, proyecciones `W_Q`, `W_K`, `W_V` (matrices `8×4`) y calcula la atención. Usa `rng = np.random.default_rng(SEED)`.

Como los pesos son aleatorios, el mapa **no** tiene significado lingüístico: sólo comprueba que el mecanismo funciona.

In [ ]:
palabras = ["El", "animal", "no", "cruzó", "la", "calle"]
d_model, d_k = 8, 4
rng = np.random.default_rng(SEED)

# TODO 1: embeddings aleatorios E con forma (6, d_model) usando rng.normal
E = ...
# TODO 2: tres matrices de proyección Wq, Wk, Wv con forma (d_model, d_k)
Wq, Wk, Wv = ..., ..., ...
# TODO 3: proyecta los embeddings (E @ W) para obtener Qs, Ks, Vs
Qs, Ks, Vs = ..., ..., ...

# TODO 4: calcula la atención con tu función
salida6, pesos6 = ...
assert salida6.shape == (6, 4) and pesos6.shape == (6, 6), "Revisa las formas"
assert np.allclose(pesos6.sum(axis=1), 1.0)
print("✓ formas correctas")

mapa_calor(pesos6, palabras, palabras, "Atención con proyecciones aleatorias")
plt.xlabel("clave")
plt.ylabel("consulta")
plt.tight_layout()
plt.show()

### A.3 — Máscara causal

En un LLM (decodificador) cada token **sólo puede mirar a los anteriores y a sí mismo**. Se logra con una máscara triangular: antes del softmax, los puntajes de las posiciones futuras se dejan en `-inf`, de modo que su peso queda exactamente en 0.

1. Escribe `mascara_causal(n)` (matriz booleana `n×n`, `True` en la diagonal y bajo ella).
2. Calcula la atención de la frase de 6 tokens con la máscara y comprueba que el triángulo superior es 0 y las filas siguen sumando 1.

In [ ]:
def mascara_causal(n):
    """Matriz booleana (n, n): True donde el token i puede mirar al token j (j <= i)."""
    # TODO: pista -> np.tril sobre una matriz de unos booleana
    ...


mascara = mascara_causal(len(palabras))
# TODO: calcula la atención de la frase de 6 tokens usando la máscara
salida_c, pesos_c = ...

assert np.allclose(np.triu(pesos_c, k=1), 0.0), "El triángulo superior debe ser 0"
assert np.allclose(pesos_c.sum(axis=1), 1.0), "Las filas deben seguir sumando 1"
assert np.isclose(pesos_c[0, 0], 1.0), "El primer token sólo puede mirarse a sí mismo"
print("✓ Máscara causal correcta")

fig, ejes = plt.subplots(1, 2, figsize=(9, 4))
mapa_calor(pesos6, palabras, palabras, "Sin máscara", ax=ejes[0], anotar=False, vmax=1)
mapa_calor(pesos_c, palabras, palabras, "Con máscara causal", ax=ejes[1], anotar=False, vmax=1)
plt.tight_layout()
plt.show()

### A.4 — ¿Por qué se divide por √d_k?

Con `d_k = 512`, genera `Q` y `K` aleatorios de 6 tokens (`rng.normal`) y calcula los pesos de atención **con** y **sin** la división por `√d_k`. Compara:

- la desviación estándar de los puntajes `Q·Kᵀ`,
- el peso máximo promedio por fila,
- la entropía promedio de las filas (`entropia`, ya definida).

In [ ]:
def entropia(p):
    """Entropía promedio (en nats) de las filas de una matriz de pesos."""
    return float(-(p * np.log(p + 1e-12)).sum(axis=-1).mean())


rng = np.random.default_rng(0)
d_k_grande = 512
Qg = rng.normal(size=(6, d_k_grande))
Kg = rng.normal(size=(6, d_k_grande))

# TODO 1: puntajes crudos Q·Kᵀ (sin dividir)
puntajes_crudos = ...
# TODO 2: pesos con softmax SIN escalar y CON la división por √d_k
sin_escala = ...
con_escala = ...

# TODO 3: imprime la desviación estándar de los puntajes, el peso máximo promedio por fila
# y la entropía promedio de cada versión
...

fig, ejes = plt.subplots(1, 2, figsize=(9, 4))
mapa_calor(sin_escala, anotar=False, ax=ejes[0], titulo="Sin dividir por √d_k", vmax=1)
mapa_calor(con_escala, anotar=False, ax=ejes[1], titulo="Dividiendo por √d_k", vmax=1)
plt.tight_layout()
plt.show()

**Pregunta:** ¿Qué pasa con los pesos si se omite la división por √d_k y d_k es grande (por ejemplo 512)?

**Respuesta:** (escribe aquí)

## Parte B — Multicabeza y codificación posicional (15 min)

### B.1 — Atención de 2 cabezas

La atención multicabeza reparte el vector de cada token en `h` trozos. Cada cabeza hace su propia atención sobre su trozo (subespacio más pequeño) y los resultados se **concatenan** y se proyectan con una matriz de salida `W_o`.

Implementa `atencion_multicabeza(X, Wq, Wk, Wv, Wo, n_cabezas=2)` dividiendo las columnas de `Q`, `K` y `V` en mitades y concatenando las salidas. Con `d_model = 8` y 2 cabezas, cada una trabaja con `d_k = 4`.

In [ ]:
def atencion_multicabeza(X, Wq, Wk, Wv, Wo, n_cabezas=2):
    """Atención multicabeza. X: (n, d_model); W*: (d_model, d_model).

    Devuelve (salida (n, d_model), pesos (n_cabezas, n, n)).
    """
    n, d_model = X.shape
    d_cabeza = d_model // n_cabezas
    # TODO 1: proyecta X con Wq, Wk y Wv -> Q, K, V de forma (n, d_model)
    Q, K, V = ..., ..., ...
    salidas, todos_pesos = [], []
    for h in range(n_cabezas):
        # TODO 2: toma las columnas de la cabeza h (slice de ancho d_cabeza) de Q, K y V
        #         y calcula su atención con la función `atencion`
        ...
    # TODO 3: concatena las salidas de las cabezas por columnas y proyecta con Wo
    salida = ...
    return salida, np.stack(todos_pesos)

In [ ]:
rng = np.random.default_rng(SEED)
d_model = 8
Wq2, Wk2, Wv2, Wo2 = (rng.normal(size=(d_model, d_model)) / np.sqrt(d_model) for _ in range(4))

salida_mh, pesos_mh = atencion_multicabeza(E, Wq2, Wk2, Wv2, Wo2, n_cabezas=2)

assert salida_mh.shape == (6, 8), "La salida debe volver a (n, d_model)"
assert pesos_mh.shape == (2, 6, 6), "Se esperan los pesos de 2 cabezas"
assert np.allclose(pesos_mh.sum(axis=-1), 1.0)
assert not np.allclose(pesos_mh[0], pesos_mh[1]), "Las cabezas deberían mirar patrones distintos"
print("✓ Multicabeza correcta: cada cabeza produce su propio mapa de atención")

fig, ejes = plt.subplots(1, 2, figsize=(9, 4))
for h in range(2):
    mapa_calor(pesos_mh[h], palabras, palabras, f"Cabeza {h + 1}", ax=ejes[h], anotar=False, vmax=1)
plt.tight_layout()
plt.show()

### B.2 — Codificación posicional sinusoidal

La atención trata la secuencia como un conjunto: sin información de posición, *«perro muerde hombre»* y *«hombre muerde perro»* se ven iguales. La codificación posicional original suma al embedding un vector que depende de la posición:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Implementa `codificacion_posicional(n_pos, d_model)` y muéstrala como imagen para 20 posiciones y 16 dimensiones.

In [ ]:
def codificacion_posicional(n_pos, d_model):
    """Matriz (n_pos, d_model) con senos en las columnas pares y cosenos en las impares."""
    pos = np.arange(n_pos)[:, None]             # (n_pos, 1)
    i = np.arange(0, d_model, 2)[None, :]       # (1, d_model/2)
    # TODO 1: ángulos = pos / 10000^(i / d_model)
    angulos = ...
    PE = np.zeros((n_pos, d_model))
    # TODO 2: senos en las columnas pares (0::2) y cosenos en las impares (1::2)
    ...
    return PE

In [ ]:
PE = codificacion_posicional(20, 16)

assert PE.shape == (20, 16)
assert np.allclose(PE[0, 0::2], 0.0) and np.allclose(PE[0, 1::2], 1.0), "En la posición 0: senos = 0 y cosenos = 1"
assert np.all(np.abs(PE) <= 1.0 + 1e-12)
print("✓ Codificación posicional correcta")

fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(PE, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xlabel("dimensión del embedding")
ax.set_ylabel("posición en la secuencia")
ax.set_title("Codificación posicional sinusoidal (20 × 16)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### B.3 — ¿Para qué sirve sumarla?

Un mismo token repetido 4 veces tiene la misma consulta y la misma clave en todas las posiciones, así que la atención lo reparte **por igual** (0,25 cada uno). Al sumar `PE`, cada copia pasa a ser distinguible.

In [ ]:
token_repetido = np.tile(E[0], (4, 1))                 # el mismo token 4 veces
_, w_sin = atencion(token_repetido, token_repetido, token_repetido)
x_pos = token_repetido + codificacion_posicional(4, d_model)
_, w_con = atencion(x_pos, x_pos, x_pos)

print("Sin posición, pesos de la primera fila:", np.round(w_sin[0], 3))
print("Con posición, pesos de la primera fila:", np.round(w_con[0], 3))
assert np.allclose(w_sin, 0.25), "Sin posición, todos los pesos deberían ser iguales"
assert not np.allclose(w_con, 0.25), "Con posición, los pesos deberían diferir"

**Pregunta:** ¿Por qué la atención por sí sola no distingue «perro muerde hombre» de «hombre muerde perro», y cómo lo resuelve la codificación posicional?

**Respuesta:** (escribe aquí)

## Parte C — Un modelo real: las probabilidades del siguiente token (35 min)

En Colab se descarga `distilgpt2` (un GPT-2 reducido de unos 82 millones de parámetros; alternativa más grande: `gpt2`). La celda de utilidades también incluye un reemplazo local con pesos aleatorios que sólo sirve para validar el código **sin conexión**; si lo ves activarse en Colab, revisa la conexión a internet: sus resultados no tienen sentido lingüístico.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

NOMBRE_MODELO = "distilgpt2"   # alternativa: "gpt2"


def _modelo_reemplazo():
    """Modelo GPT-2 pequeño con pesos ALEATORIOS y tokenizador local. Sólo para validar el código sin red."""
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    from transformers import PreTrainedTokenizerFast, GPT2Config, GPT2LMHeadModel
    corpus = ["El animal no cruzó la calle porque estaba muy cansado",
              "La capital de Chile es Santiago",
              "Machine learning es genial y me gusta mucho aprender",
              "Machine learning is great and I really enjoy learning",
              "El premio Nobel de Física de 1987 fue ganado por"] * 20
    tk = Tokenizer(models.BPE(unk_token="<unk>"))
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    entrenador = trainers.BpeTrainer(vocab_size=500, special_tokens=["<unk>", "<|endoftext|>"],
                                     initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tk.train_from_iterator(corpus, entrenador)
    tok = PreTrainedTokenizerFast(tokenizer_object=tk, eos_token="<|endoftext|>", unk_token="<unk>")
    cfg = GPT2Config(vocab_size=len(tok), n_positions=128, n_embd=64, n_layer=2, n_head=4,
                     bos_token_id=1, eos_token_id=1)
    torch.manual_seed(SEED)
    return tok, GPT2LMHeadModel(cfg)


def cargar_modelo(nombre=NOMBRE_MODELO):
    """Descarga el modelo real; sin red usa el reemplazo local. Devuelve (tokenizador, modelo, es_real)."""
    try:
        tok = AutoTokenizer.from_pretrained(nombre)
        modelo = AutoModelForCausalLM.from_pretrained(nombre, attn_implementation="eager")
        es_real = True
    except Exception as error:  # sin conexión
        print(f"⚠ No se pudo descargar «{nombre}» ({type(error).__name__}). "
              "Usando un modelo con pesos ALEATORIOS sólo para validar el código.")
        tok, modelo = _modelo_reemplazo()
        es_real = False
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    try:                                        # necesario para poder leer los pesos de atención (Parte D)
        modelo.set_attn_implementation("eager")
    except AttributeError:                      # versiones antiguas de transformers
        modelo.config._attn_implementation = "eager"
    modelo.eval()
    return tok, modelo, es_real


tok, modelo, MODELO_REAL = cargar_modelo()
print("modelo real:", MODELO_REAL, "| parámetros:", sum(p.numel() for p in modelo.parameters()))

### C.1 — Tokenización

Observa cómo se parte una frase en español y su equivalente en inglés con `tok.tokenize`. Compara el número de tokens.

In [ ]:
frases = {
    "español": "Machine learning es genial y me gusta mucho aprender",
    "inglés": "Machine learning is great and I really enjoy learning",
}
# TODO: para cada frase, tokeniza con tok.tokenize(...) e imprime el idioma, el número de tokens y los trozos
for idioma, frase in frases.items():
    ...

**Pregunta:** ¿Cuál de las dos frases usa más tokens? ¿Qué implica para el costo y la ventana de contexto en español?

**Respuesta:** (escribe aquí)

### C.2 — Probabilidades del siguiente token

Para la frase *«La capital de Chile es»*:

1. Obtén los `logits` del **último** token (`modelo(ids).logits[0, -1]`).
2. Aplica softmax y muestra los 10 tokens más probables con su probabilidad (`torch.topk`).

Implementa `probs_siguiente_token(modelo, tok, texto, k=10)`, que devuelve una lista de pares `(token, probabilidad)`.

In [ ]:
def probs_siguiente_token(modelo, tok, texto, k=10):
    """Devuelve los k tokens más probables para continuar `texto` como lista de (token, probabilidad)."""
    ids = tok(texto, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        # TODO 1: logits de la ÚLTIMA posición de la secuencia (modelo(ids).logits[0, -1])
        logits = ...
    # TODO 2: convierte los logits en probabilidades con softmax
    probs = ...
    # TODO 3: torch.topk(probs, k) y devuelve [(tok.decode([id]), prob), ...]
    ...

In [ ]:
prompt = "La capital de Chile es"
top10 = probs_siguiente_token(modelo, tok, prompt, k=10)

for token, p in top10:
    print(f"{token!r:>14}  {p:6.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh([repr(t) for t, _ in top10][::-1], [p for _, p in top10][::-1], color=AZUL)
ax.set_xlabel("probabilidad")
ax.set_title(f"Siguiente token tras «{prompt}»")
plt.tight_layout()
plt.show()

### C.3 — Muestreo: temperatura, top-k y top-p

El modelo entrega una distribución; **cómo se elige** el token de ella cambia el texto.

- **Temperatura `T`:** se dividen los logits por `T` antes del softmax. `T` baja → casi siempre el más probable; `T` alta → distribución más plana y más sorpresas.
- **top-k:** sólo compiten los `k` tokens más probables.
- **top-p:** sólo compite el menor conjunto de tokens cuya probabilidad acumulada llega a `p`.

Implementa `muestrear(logits, temperatura, top_k, top_p, generador)` y luego `generar(...)`, que repite el proceso token a token.

In [ ]:
def muestrear(logits, temperatura=1.0, top_k=None, top_p=None, generador=None):
    """Elige el id de un token a partir de los logits (tensor 1D)."""
    # TODO 1: divide los logits por la temperatura (clone() primero para no modificar el original)
    logits = ...
    if top_k is not None:
        # TODO 2: deja en -inf todo logit menor que el k-ésimo más grande (torch.topk)
        ...
    if top_p is not None:
        # TODO 3: ordena de mayor a menor, calcula la probabilidad acumulada y descarta los tokens
        #         cuya probabilidad acumulada ANTES de ellos ya superó top_p; luego deshaz el orden
        ...
    # TODO 4: softmax y torch.multinomial(probs, 1, generator=generador) -> id del token elegido
    ...


def generar(modelo, tok, texto, n_tokens=15, temperatura=1.0, top_k=None, top_p=None, semilla=0):
    """Genera `n_tokens` tokens nuevos, de a uno, y devuelve el texto completo."""
    g = torch.Generator().manual_seed(semilla)
    ids = tok(texto, return_tensors="pt")["input_ids"]
    for _ in range(n_tokens):
        # TODO 5: logits de la última posición -> muestrear -> concatenar el id nuevo a `ids`
        ...
    return tok.decode(ids[0])

In [ ]:
# Autocomprobación con logits inventados (no usa el modelo)
l = torch.tensor([2.0, 1.0, 0.1, -1.0])           # softmax ≈ [0.64, 0.24, 0.10, 0.03]
g = torch.Generator().manual_seed(0)
assert muestrear(l, temperatura=0.01, generador=g) == 0, "Con T muy baja debe elegirse el más probable"
assert all(muestrear(l, top_k=1, generador=g) == 0 for _ in range(30)), "top_k=1 equivale a elegir el máximo"
assert all(muestrear(l, top_p=0.5, generador=g) == 0 for _ in range(30)), "Con p=0.5 sólo cabe el primer token"
frec = np.bincount([muestrear(l, generador=g) for _ in range(4000)], minlength=4) / 4000
assert abs(frec[0] - 0.64) < 0.04, f"La frecuencia empírica del token 0 debería ser ≈ 0.64 y fue {frec[0]:.2f}"
print("✓ muestrear() se comporta como se espera; frecuencias con T=1:", np.round(frec, 2))

In [ ]:
prompt = "La capital de Chile es"
# TODO: para T en (0.2, 1.0, 1.8) genera 5 continuaciones de 12 tokens (semillas 0..4) e imprímelas
for T in (0.2, 1.0, 1.8):
    ...

In [ ]:
# TODO: repite con top_k=50 y luego con top_p=0.9 (T = 1.0), 3 semillas cada uno
...

**Pregunta:** Compara la diversidad y la coherencia de las continuaciones con temperatura 0,2, 1,0 y 1,8. ¿Cuándo te interesaría cada una?

**Respuesta:** (escribe aquí)

### C.4 — Generación con `model.generate`

La librería ya trae todo lo anterior empaquetado. Genera 40 tokens nuevos con `do_sample=True`, `temperature=0.8`, `top_p=0.9`, con tres semillas distintas (`torch.manual_seed`).

In [ ]:
prompt = "Machine learning es"
# TODO: con tres semillas (torch.manual_seed), llama a modelo.generate(ids, max_new_tokens=40,
# do_sample=True, temperature=0.8, top_p=0.9, pad_token_id=tok.eos_token_id) y decodifica el resultado
for semilla in range(3):
    ...

### C.5 — Alucinación

Pide al modelo que complete un dato difícil: *«El premio Nobel de Física de 1987 fue ganado por»*. Genera una respuesta determinista (`do_sample=False`) y tres muestreadas. ¿Suena segura? ¿Es correcta?

In [ ]:
prompt = "El premio Nobel de Física de 1987 fue ganado por"
# TODO: una generación voraz (do_sample=False, 30 tokens nuevos) y tres muestreadas (T = 1.0, semillas 0..2)
...

**Pregunta:** ¿Qué responde el modelo, con qué seguridad lo dice y qué te dice eso sobre cómo está entrenado?

**Respuesta:** (escribe aquí)

## Parte D — Mirar la atención real (20 min)

Con `output_attentions=True` el modelo devuelve, para cada capa, un tensor `(lote, cabezas, n, n)` con los pesos de atención de cada cabeza. (Eso funciona porque la celda de carga fijó la implementación `eager` de la atención.)

1. Obtén los pesos de la frase *«El animal no cruzó la calle porque estaba muy cansado»* y dibuja el mapa de una capa y una cabeza.
2. Busca una cabeza que atienda mayormente al **token anterior** y otra que se concentre en el **primer token**.
3. Cuenta los parámetros y estima cuántos están en la atención y cuántos en las capas densas.

*Nota:* en los rótulos, `·` marca un espacio y trozos como `Ã³` son los bytes de una letra con tilde (el tokenizador de GPT-2 trabaja con bytes).

In [ ]:
FRASE_D = "El animal no cruzó la calle porque estaba muy cansado"


def atenciones(modelo, tok, texto):
    """Devuelve (pesos (capas, cabezas, n, n) como arreglo NumPy, lista de tokens)."""
    ids = tok(texto, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        # TODO 1: llama al modelo con output_attentions=True
        salida = ...
    # TODO 2: apila salida.attentions (una por capa), quita la dimensión de lote [:, 0] y pasa a NumPy
    att = ...
    tokens = [t.replace("Ġ", "·") for t in tok.convert_ids_to_tokens(ids[0])]
    return att, tokens


att, tokens = atenciones(modelo, tok, FRASE_D)
n_capas, n_cabezas = att.shape[:2]
print(f"{n_capas} capas × {n_cabezas} cabezas | {len(tokens)} tokens: {tokens}")

In [ ]:
capa, cabeza = 0, 0       # prueba otras combinaciones
mapa_calor(att[capa, cabeza], tokens, tokens, f"Capa {capa}, cabeza {cabeza}", anotar=False, vmax=1)
plt.xlabel("clave (token al que se mira)")
plt.ylabel("consulta (token que mira)")
plt.tight_layout()
plt.show()

**Cabezas especializadas.** Para cada par `(capa, cabeza)` calcula dos puntajes promedio: cuánta atención recibe el **token anterior** (la diagonal bajo la principal) y cuánta recibe el **primer token** (columna 0, ignorando la primera fila). Muéstralos como mapas de calor `capas × cabezas` e identifica el mejor candidato de cada tipo.

In [ ]:
def puntajes_cabezas(att):
    """Devuelve (puntaje_anterior, puntaje_primero), cada uno de forma (capas, cabezas)."""
    # TODO 1: atención media que cada consulta i (i >= 1) pone en el token i-1
    #         pista: np.diagonal(att, offset=-1, axis1=-2, axis2=-1) y promedio sobre el último eje
    anterior = ...
    # TODO 2: atención media que las consultas 1.. ponen en el token 0
    primero = ...
    return anterior, primero


anterior, primero = puntajes_cabezas(att)
assert anterior.shape == (n_capas, n_cabezas) and primero.shape == (n_capas, n_cabezas)
mejor_ant = np.unravel_index(anterior.argmax(), anterior.shape)
mejor_pri = np.unravel_index(primero.argmax(), primero.shape)
print(f"Cabeza que más mira al token anterior: capa {mejor_ant[0]}, cabeza {mejor_ant[1]} ({anterior.max():.2f})")
print(f"Cabeza que más mira al primer token : capa {mejor_pri[0]}, cabeza {mejor_pri[1]} ({primero.max():.2f})")

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for ax, M_, titulo in ((ejes[0], anterior, "Atención al token anterior"), (ejes[1], primero, "Atención al primer token")):
    im = ax.imshow(M_, cmap=CMAP, vmin=0, vmax=1, aspect="auto")
    ax.set_title(titulo)
    ax.set_xlabel("cabeza")
    ax.set_ylabel("capa")
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# TODO (extra): dibuja con mapa_calor los mapas de atención de las dos cabezas encontradas

**Pregunta:** Describe lo que observas en las dos cabezas elegidas. ¿Por qué tiene sentido que haya cabezas tan simples?

**Respuesta:** (escribe aquí)

**Parámetros.** Cuenta los parámetros totales y cuántos pertenecen a la atención (`.attn.` en el nombre), a las capas densas (`.mlp.`) y a los embeddings (`wte`, `wpe`).

In [ ]:
def contar_parametros(modelo):
    """Devuelve un diccionario con el total y su reparto por tipo de componente."""
    nombres = list(modelo.named_parameters())
    # TODO: suma los parámetros (p.numel()) según el nombre: ".attn." (atención), ".mlp." (capas densas),
    #       "wte"/"wpe" (embeddings); el resto es normalización
    ...


reparto = contar_parametros(modelo)
for nombre, valor in reparto.items():
    print(f"{nombre:24s} {valor:>12,d}")

**Pregunta:** ¿Qué porcentaje de los parámetros está en la atención y cuál en las capas densas? ¿Es lo que esperabas?

**Respuesta:** (escribe aquí)

## Parte E — Clasificación con un modelo preentrenado (opcional, 10 min)

Un modelo ya preentrenado y ajustado puede clasificar texto sin entrenar nada. Usa `pipeline("sentiment-analysis")` con un modelo multilingüe pequeño, evalúa **10 frases propias** (en español) etiquetadas por ti como `positive`, `neutral` o `negative`, y calcula la exactitud (herramienta de la clase 9).

In [ ]:
from transformers import pipeline

clasificador = pipeline("sentiment-analysis",
                        model="cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual")

# TODO 1: escribe 10 frases propias con su etiqueta ("positive", "neutral" o "negative")
datos = [
    ("Me encantó la clase de hoy, aprendí mucho", "positive"),
    # ...
]
# TODO 2: predice cada frase (clasificador(texto)[0]["label"].lower()) y calcula la exactitud
...

**Pregunta:** ¿En qué frases se equivocó el modelo y por qué crees que ocurrió?

**Respuesta:** (escribe aquí)

## Entregable

Notebook con las **Partes A–D ejecutadas** (la E es opcional) y un párrafo de **6–8 líneas** respondiendo:

> *Explica con tus palabras cómo un LLM decide la siguiente palabra y por qué puede equivocarse con total seguridad.*

**Entrega por AULA-USM.**

**Párrafo (6–8 líneas):** (escribe aquí)